# Grid search para Random Forest, RBF-SVC y XGBoost

Este notebook realiza una busqueda de hiperparametros para los tres modelos clasicos mas relevantes del proyecto:

- `random_forest`
- `rbf_svc`
- `xgboost`

La validacion usa `StratifiedGroupKFold`, de forma que las ventanas de un mismo sujeto no aparecen a la vez en train y test. Esto mantiene el criterio cross-subject del entrenamiento principal.

La configuracion de epochs y features esta alineada con `scripts/train_ml.py`: `epoch_size=1920`, `step_size=960` y `nperseg=960`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedGroupKFold

PROJECT_ROOT = Path.cwd().parent
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))


## Funciones auxiliares

In [ ]:
def add_pipeline_prefix(estimator, base_grid):
    if hasattr(estimator, "named_steps"):
        last_step_name = list(estimator.named_steps.keys())[-1]
        return {f"{last_step_name}__{k}": v for k, v in base_grid.items()}
    return base_grid


def tune_models(
    models,
    param_grids,
    X,
    y,
    groups,
    model_names=("random_forest", "rbf_svc", "xgboost"),
    cv_splits=5,
    scoring="balanced_accuracy",
    search_type="grid",   # "grid" o "random"
    n_iter=12,             # solo para random search
    n_jobs=1,
    verbose=2,
):
    cv = StratifiedGroupKFold(
        n_splits=cv_splits,
        shuffle=True,
        random_state=42,
    )

    results = []
    best_estimators = {}

    for model_name in model_names:
        print(f"\n=== TUNING: {model_name} ===")

        model = clone(models[model_name])
        grid = add_pipeline_prefix(model, param_grids[model_name])

        if search_type == "grid":
            search = GridSearchCV(
                estimator=model,
                param_grid=grid,
                scoring=scoring,
                cv=cv,
                n_jobs=n_jobs,
                refit=True,
                verbose=verbose,
            )
        elif search_type == "random":
            search = RandomizedSearchCV(
                estimator=model,
                param_distributions=grid,
                n_iter=n_iter,
                scoring=scoring,
                cv=cv,
                n_jobs=n_jobs,
                refit=True,
                verbose=verbose,
                random_state=42,
            )
        else:
            raise ValueError("search_type debe ser 'grid' o 'random'")

        search.fit(X, y, groups=groups)

        best_estimators[model_name] = search.best_estimator_
        results.append({
            "Modelo": model_name,
            "Scoring": scoring,
            "BestScore": search.best_score_,
            "BestParams": search.best_params_,
        })

        print("Best score:", search.best_score_)
        print("Best params:", search.best_params_)

    return pd.DataFrame(results), best_estimators


## Importar modelos y definir grids

Los grids son moderados para que el notebook sea ejecutable en local. Si quieres una busqueda mas rapida, cambia `search_type="random"` en la celda final.


In [ ]:
from scripts.pipeline import get_models

models = get_models()

param_grids = {
    "random_forest": {
        "n_estimators": [100, 200, 500],
        "max_depth": [None, 10, 20],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", 0.5],
        "class_weight": ["balanced"],
    },
    "rbf_svc": {
        "C": [1, 10, 50],
        "gamma": ["scale", 0.01, 0.001],
        "class_weight": ["balanced"],
    },
    "xgboost": {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 4, 5],
        "learning_rate": [0.03, 0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    },
}


In [ ]:
from pathlib import Path

from scripts.data_load import load_dataset
from scripts.preprocessing import preprocess_dataset
from scripts.epochs import create_epochs
from scripts.features import extract_epoch_features
from scripts.spectral_features import extract_spectral_features

CSV_PATH = PROJECT_ROOT / "data" / "adhdata.csv"

EPOCH_SIZE = 1920
STEP_SIZE = 960
SFREQ = 128
NPERSEG = 960

df = load_dataset(CSV_PATH)
df_clean, eeg_cols = preprocess_dataset(df)

x_epochs, y_epochs, groups_epochs = create_epochs(
    df=df_clean,
    eeg_columns=eeg_cols,
    label_column="Class",
    group_column="ID",
    epoch_size=EPOCH_SIZE,
    step_size=STEP_SIZE,
)

x_time = extract_epoch_features(x_epochs, eeg_cols)

x_spectral = extract_spectral_features(
    x_epochs=x_epochs,
    channel_names=eeg_cols,
    sfreq=SFREQ,
    nperseg=NPERSEG,
)

X_features = pd.concat(
    [x_time.reset_index(drop=True), x_spectral.reset_index(drop=True)],
    axis=1,
)

print("X_epochs:", x_epochs.shape)
print("X_time:", x_time.shape)
print("X_spectral:", x_spectral.shape)
print("X_features:", X_features.shape)
print("y_epochs:", y_epochs.shape)
print("groups_epochs:", groups_epochs.shape)


## Ejecutar grid search

Aqui se tunearan `random_forest`, `rbf_svc` y `xgboost` con el mismo esquema cross-subject del entrenamiento principal.

Si tarda demasiado, cambia `search_type="grid"` por `search_type="random"` y ajusta `n_iter`.


In [ ]:
tuning_results_df, best_estimators = tune_models(
    models=models,
    param_grids=param_grids,
    X=X_features,
    y=y_epochs,
    groups=groups_epochs,
    model_names=("random_forest", "rbf_svc", "xgboost"),
    cv_splits=5,
    scoring="balanced_accuracy",
    search_type="grid",
    n_iter=20,
    n_jobs=1,
    verbose=2,
)

tuning_results_df


## Scores

- `BestScore`: mejor balanced accuracy media en CV.
- `BestParams`: hiperparametros ganadores.
- `best_estimators`: modelos ya ajustados con la mejor configuracion.

Puedes trasladar los mejores parametros a `scripts/pipeline.py` si superan claramente al baseline de `train_ml.py`.
